# EDA - Credit Score

Business Problem:

> "Will this customer default on their credit line in the next 2 years?"

## Data Dictionary



Install packages

In [1]:
!uv pip install -q \
        pandas \
        pyarrow \
        boto3

Import packages

In [2]:
import pandas as pd
import boto3
import io

Inspect dataset

In [3]:
endpoint = "http://localstack:4566"
bucket_name = "data-lake"
prefix = "silver/credit_risk/cleaned/ingestion_date=2026-03-14/"

s3_client = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1"
)


response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
parquet_keys = [
    obj['Key'] for obj in response.get('Contents', []) 
    if obj['Key'].endswith('.parquet')
]


dfs = []

for key in parquet_keys:
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    dfs.append(pd.read_parquet(io.BytesIO(obj['Body'].read())))

df = pd.concat(dfs, ignore_index=True)

df.head()

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
0,1,0.880371,43,0,0.461324,7523.0,6,0,1,0,1.0
1,0,0.022421,72,0,1004.000000,NaN,6,0,1,0,0.0
2,0,0.558882,49,0,0.487112,9077.0,18,0,1,0,3.0
3,0,0.145783,61,0,1211.000000,NaN,8,0,1,0,0.0
4,0,0.207405,54,0,0.390859,8685.0,11,0,1,0,2.0


In [4]:
df.shape

(149391, 11)

In [5]:
df.columns

Index(['serious_dlqin2yrs', 'revolving_utilization_of_unsecured_lines', 'age',
       'number_of_time30_59_days_past_due_not_worse', 'debt_ratio',
       'monthly_income', 'number_of_open_credit_lines_and_loans',
       'number_of_times90_days_late', 'number_real_estate_loans_or_lines',
       'number_of_time60_89_days_past_due_not_worse', 'number_of_dependents'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149391 entries, 0 to 149390
Data columns (total 11 columns):
 #   Column                                       Non-Null Count   Dtype  
---  ------                                       --------------   -----  
 0   serious_dlqin2yrs                            149391 non-null  int32  
 1   revolving_utilization_of_unsecured_lines     149391 non-null  float32
 2   age                                          149391 non-null  int32  
 3   number_of_time30_59_days_past_due_not_worse  149391 non-null  int32  
 4   debt_ratio                                   149391 non-null  float32
 5   monthly_income                               120170 non-null  float32
 6   number_of_open_credit_lines_and_loans        149391 non-null  int32  
 7   number_of_times90_days_late                  149391 non-null  int32  
 8   number_real_estate_loans_or_lines            149391 non-null  int32  
 9   number_of_time60_89_days_past_due_not_worse  149391 non-null  int32  


In [7]:
df.describe()

,serious_dlqin2yrs,revolving_utilization_of_unsecured_lines,age,number_of_time30_59_days_past_due_not_worse,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_of_times90_days_late,number_real_estate_loans_or_lines,number_of_time60_89_days_past_due_not_worse,number_of_dependents
count,149391.000000,149391.000000,149391.000000,149391.000000,149391.000000,1.201700e+05,149391.000000,149391.000000,149391.000000,149391.000000,145563.000000
mean,0.066999,6.071087,52.306237,0.393886,354.436707,6.675098e+03,8.480892,0.238120,1.022391,0.212503,0.759863
std,0.250021,250.263672,14.725962,3.852953,2041.843506,1.438958e+04,5.136515,3.826165,1.130196,3.810523,1.116141
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.030132,41.000000,0.000000,0.177441,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154235,52.000000,0.000000,0.368234,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.556494,63.000000,0.000000,0.875279,8.250000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


## Target Variable

In [8]:
df["serious_dlqin2yrs"].value_counts(normalize=True)

serious_dlqin2yrs
0    0.933001
1    0.066999
Name: proportion, dtype: float64